# Factorized Graded Response Model — PROMIS Neuropathic Pain

Fits a **FactorizedGRModel** to multiple pain-related PROMIS banks
from the Neuropathic Pain adult dataset (doi:10.7910/DVN/TJ9MNM).

Only domains with ≥10 items are retained. The SAS data file is
read via pandas.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['TQDM_DISABLE'] = '1'
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from plot_helpers import (plot_loss_comparison, plot_forest_discriminations,
                          plot_ability_scatter, plot_ability_distributions,
                          plot_thresholds, plot_individual_abilities,
                          plot_imputation_weights_pcolormesh)

## 1. Load Data (Multiple Domains)

In [ ]:
from bayesianquilts.data.promis_neuropathic_pain import (
    get_multidomain_data, item_keys, response_cardinality, DOMAINS,
)

df, num_people, scale_indices = get_multidomain_data(
    polars_out=True, min_items=10,
)

print(f"\nDataset: {num_people} people, {len(item_keys)} total items, "
      f"{len(scale_indices)} domains")
print(f"Response categories: {response_cardinality} (0-{response_cardinality - 1})")
for domain, indices in scale_indices.items():
    print(f"  {domain}: {len(indices)} items")
df.head()

In [ ]:
SUBSAMPLE_N = num_people
sub_df = df
print(f"Using full dataset: N = {SUBSAMPLE_N}")

## 2. Prepare Data

In [ ]:
def make_data_dict(dataframe):
    data = {}
    for col in dataframe.columns:
        arr = dataframe[col].to_numpy().astype(np.float64)
        data[col] = arr
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

batch = make_data_dict(sub_df)

# Check for missing/invalid values
n_bad = sum(
    np.sum(np.isnan(batch[k]) | (batch[k] < 0) | (batch[k] >= response_cardinality))
    for k in item_keys
)
print(f"Bad/missing values: {n_bad}")

BATCH_SIZE = 256
steps_per_epoch = int(np.ceil(SUBSAMPLE_N / BATCH_SIZE))
print(f"N: {SUBSAMPLE_N}, Batch size: {BATCH_SIZE}, Steps per epoch: {steps_per_epoch}")

def data_factory():
    indices = np.arange(SUBSAMPLE_N)
    np.random.shuffle(indices)
    for start in range(0, SUBSAMPLE_N, BATCH_SIZE):
        end = min(start + BATCH_SIZE, SUBSAMPLE_N)
        idx_batch = indices[start:end]
        yield {k: v[idx_batch] for k, v in batch.items()}

## 3. Fit Baseline FactorizedGRModel (Ignorable Missingness)

Each domain loads onto its own latent dimension. The model estimates
per-domain abilities and item discriminations.

In [ ]:
from bayesianquilts.irt.factorizedgrm import FactorizedGRModel

NUM_EPOCHS = 200
SNAPSHOT_EPOCH = 50

model_baseline = FactorizedGRModel(
    scale_indices=scale_indices,
    discrimination_prior_scale=2.0,
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
)

res_baseline = model_baseline.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
    snapshot_epoch=SNAPSHOT_EPOCH,
)

losses_baseline = res_baseline[0]
snapshot_params = res_baseline[2] if len(res_baseline) > 2 else None
print(f"Baseline final loss: {losses_baseline[-1]:.2f}")

In [ ]:
model_baseline.save_to_disk('grm_baseline')

In [ ]:
def calibrate_manually(model, n_samples=32, seed=42):
    try:
        surrogate = model.surrogate_distribution_generator(model.params)
        key = jax.random.PRNGKey(seed)
        samples = surrogate.sample(n_samples, seed=key)
        expectations = {k: jnp.mean(v, axis=0) for k, v in samples.items()}
        model.calibrated_expectations = expectations
        model.surrogate_sample = samples
    except KeyError as e:
        print(f"  Warning: surrogate sampling failed ({e}), using point estimates")
        point_estimates = {}
        for key_name, value in model.params.items():
            parts = key_name.split('\\')
            if len(parts) >= 4:
                param_name = parts[0]
                if parts[-2] == 'normal' and parts[-1] == 'loc':
                    point_estimates[param_name] = value
        model.calibrated_expectations = point_estimates

calibrate_manually(model_baseline, n_samples=32, seed=101)

## 4. Fit Pairwise Ordinal Stacking Model

In [ ]:
from bayesianquilts.imputation.pairwise_stacking import PairwiseOrdinalStackingModel

# Convert to pandas, replacing -1 (missing marker) with NaN
pandas_df = sub_df.select(item_keys).to_pandas()
pandas_df = pandas_df.replace(-1, np.nan)
print(f"Missing values per item:\n{pandas_df.isna().sum()}")

pairwise_model = PairwiseOrdinalStackingModel(
    prior_scale=1.0,
    pathfinder_num_samples=100,
    pathfinder_maxiter=50,
    batch_size=512,
    verbose=True,
)

pairwise_model.fit(
    pandas_df,
    n_top_features=12,
    n_jobs=-1,
    seed=42,
)

In [ ]:
pairwise_model.save('pairwise_stacking_model.yaml')
pairwise_model.compute_optimal_stacking_weights()

## 4b. Fit Pairwise-Only Factorized GRM

Uses only the pairwise ordinal stacking ensemble (w=1 for all items).

In [ ]:
from bayesianquilts.imputation.mixed import PairwiseOnlyImputationModel

pairwise_imputation = PairwiseOnlyImputationModel(mice_model=pairwise_model)
print(pairwise_imputation.summary())

model_pairwise = FactorizedGRModel(
    scale_indices=scale_indices,
    discrimination_prior_scale=2.0,
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    response_cardinality=response_cardinality,
    imputation_model=pairwise_imputation,
    dtype=jnp.float64,
)

print("Warm-starting from baseline model parameters")

res_pairwise = model_pairwise.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
    initial_values=model_baseline.params,
)

losses_pairwise = res_pairwise[0]
print(f"Final pairwise loss: {losses_pairwise[-1]:.2f}")
model_pairwise.save_to_disk('grm_pairwise')

In [ ]:
from bayesianquilts.imputation.mixed import IrtMixedImputationModel

mixed_imputation = IrtMixedImputationModel(
    irt_model=model_baseline,
    mice_model=pairwise_model,
    data_factory=data_factory,
    irt_elpd_batch_size=4,
)

print(mixed_imputation.summary())

## 5. Fit Mixed Factorized GRM (Pairwise + IRT Imputation)

In [ ]:
all_pairwise = all(mixed_imputation.get_item_weight(k) >= 1.0 - 1e-6 for k in item_keys)

if all_pairwise:
    print("All w_IRT ≈ 0 — mixed model is identical to pairwise.")
    model_imputed = model_pairwise
    losses_imputed = losses_pairwise
else:
    n_irt = sum(1 for k in item_keys if mixed_imputation.get_item_weight(k) < 1.0 - 1e-6)
    print(f"IRT contributes to {n_irt}/{len(item_keys)} items")
    model_imputed = FactorizedGRModel(
        scale_indices=scale_indices,
        discrimination_prior_scale=2.0,
        item_keys=item_keys,
        num_people=SUBSAMPLE_N,
        response_cardinality=response_cardinality,
        imputation_model=mixed_imputation,
        dtype=jnp.float64,
    )

    print("Warm-starting from pairwise model parameters")
    res_imputed = model_imputed.fit(
        data_factory,
        batch_size=BATCH_SIZE,
        dataset_size=SUBSAMPLE_N,
        num_epochs=NUM_EPOCHS,
        steps_per_epoch=steps_per_epoch,
        learning_rate=0.0001,
        patience=10,
        initial_values=model_pairwise.params,
    )

    losses_imputed = res_imputed[0]
    if losses_imputed:
        print(f"Final mixed loss: {losses_imputed[-1]:.2f}")
    else:
        model_imputed = model_pairwise
        losses_imputed = losses_pairwise

In [ ]:
model_imputed.save_to_disk('grm_imputed')

## 6. Compare Results

In [ ]:
fig = plot_loss_comparison(losses_baseline, losses_imputed,
                          title='PROMIS Neuropathic Pain',
                          losses_pairwise=losses_pairwise)
fig.savefig('loss_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
calibrate_manually(model_pairwise, n_samples=32, seed=103)
calibrate_manually(model_imputed, n_samples=32, seed=102)

## 7. Per-Domain Ability Comparison

Compare ability estimates across domains between baseline and imputed models.

In [ ]:
# Plot per-domain ability distributions
fig, axes = plt.subplots(1, len(scale_indices), figsize=(4 * len(scale_indices), 4))
if len(scale_indices) == 1:
    axes = [axes]

for d, (ax, (domain, indices)) in enumerate(zip(axes, scale_indices.items())):
    ab_base = np.array(model_baseline.calibrated_expectations[f'abilities_{d}']).flatten()
    ab_imp = np.array(model_imputed.calibrated_expectations[f'abilities_{d}']).flatten()
    ax.hist(ab_base, bins=30, alpha=0.5, label='Baseline', density=True)
    ax.hist(ab_imp, bins=30, alpha=0.5, label='Imputed', density=True)
    ax.set_title(domain)
    ax.legend()

fig.suptitle('PROMIS Neuropathic Pain — Per-Domain Ability Distributions')
fig.tight_layout()
fig.savefig('ability_distributions.pdf', bbox_inches='tight', dpi=150)
plt.show()

## Summary

This notebook fitted a Factorized Graded Response Model to multiple
pain-related PROMIS item banks from the Neuropathic Pain adult dataset.

Each domain (Pain Interference, Pain Behavior, etc.) loads onto its own
latent dimension. Only domains with ≥10 items were retained.

Data source: Harvard Dataverse (doi:10.7910/DVN/TJ9MNM).